In [1]:
"""
==============================================================================
🚀 [Step 5] PyTorch Dataset Integration & YOLO Converter
==============================================================================
### @Author       : 한의정 (Data Engineering Lead)
### @Description  : 최종 가공된 전처리 데이터를 모델링 학습 루프(Train Loop)에 통합.
###                 Faster R-CNN / RetinaNet용 DataLoader 및 YOLO용 레이블 변환기 제공.
### @Bridge       : Modeling Team 배포용 최종 표준 데이터셋 (COCO Format 기반)
==============================================================================

📊 [Data Constraint Report] 희귀 클래스(Minority) 제약 사항
  - 현상: 데이터 1장 보유 희귀 클래스 6종 → Train 셋 전량 배정
  - 영향: Val 셋에 해당 클래스 없어 mAP 0으로 표시될 수 있음
  - 대응: Copy-Paste 증강으로 학습 수행, 최종 평가는 Confusion Matrix로 보완 예정
"""

'\n==============================================================================\n🚀 [Step 5] PyTorch Dataset Integration & YOLO Converter\n==============================================================================\n### @Author       : 한의정 (Data Engineering Lead)\n### @Description  : 최종 가공된 전처리 데이터를 모델링 학습 루프(Train Loop)에 통합.\n###                 Faster R-CNN / RetinaNet용 DataLoader 및 YOLO용 레이블 변환기 제공.\n### @Bridge       : Modeling Team 배포용 최종 표준 데이터셋 (COCO Format 기반)\n==============================================================================\n\n📊 [Data Constraint Report] 희귀 클래스(Minority) 제약 사항\n  - 현상: 데이터 1장 보유 희귀 클래스 6종 → Train 셋 전량 배정\n  - 영향: Val 셋에 해당 클래스 없어 mAP 0으로 표시될 수 있음\n  - 대응: Copy-Paste 증강으로 학습 수행, 최종 평가는 Confusion Matrix로 보완 예정\n'

In [2]:
# ============================================================
# [Cell 0] 환경 설정 — Colab / 로컬 자동 감지
# ============================================================
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    sys.path.insert(0, os.path.abspath('..'))
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../data'))

print(f"✅ 환경: {'Colab' if is_colab else '로컬'}")
print(f"✅ PROJECT: {sys.path[0]}")
print(f"✅ DATA:    {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 환경: Colab
✅ PROJECT: /content/pill_detection_project
✅ DATA:    /content/drive/MyDrive/data/초급_프로젝트/dataset


In [3]:
############################################################
# 0. 환경 설정 및 경로
############################################################
import os
import json
import platform
import pandas as pd
from collections import defaultdict

TRAIN_JSON_PATH  = os.path.join(BASE_DIR, 'train_letterbox.json')
VAL_JSON_PATH    = os.path.join(BASE_DIR, 'val_letterbox.json')
TRAIN_IMG_DIR    = os.path.join(BASE_DIR, 'letterbox_images/train')
VAL_IMG_DIR      = os.path.join(BASE_DIR, 'letterbox_images/val')

In [4]:
############################################################
# 1. 파이프라인 검증 (Sanity Check)
############################################################
def validate_coco(json_path, target_size=800):
    if not os.path.exists(json_path):
        print(f"🚨 파일을 찾을 수 없습니다: {json_path}")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    wrong_size = [img for img in coco['images']
                  if img['width'] != target_size or img['height'] != target_size]

    issues = []
    for ann in coco['annotations']:
        x, y, w, h = ann['bbox']
        if x < 0 or y < 0:                          issues.append(('negative_xy',      ann['id']))
        if w <= 0 or h <= 0:                         issues.append(('non_positive_wh',  ann['id']))
        if x + w > target_size or y + h > target_size: issues.append(('out_of_bounds', ann['id']))

    print(f"\n[{os.path.basename(json_path)}]")
    print(f"  • 이미지 수       : {len(coco['images'])}장")
    print(f"  • BBox 총 수      : {len(coco['annotations'])}개")
    print(f"  • 규격 이상 이미지 : {len(wrong_size)}장")
    print(f"  • BBox 좌표 이슈  : {len(issues)}개")

validate_coco(TRAIN_JSON_PATH)
validate_coco(VAL_JSON_PATH)


[train_letterbox.json]
  • 이미지 수       : 1792장
  • BBox 총 수      : 6196개
  • 규격 이상 이미지 : 0장
  • BBox 좌표 이슈  : 0개

[val_letterbox.json]
  • 이미지 수       : 139장
  • BBox 총 수      : 431개
  • 규격 이상 이미지 : 0장
  • BBox 좌표 이슈  : 0개


In [5]:
############################################################
# 2. Faster R-CNN / RetinaNet용 DataLoader
############################################################
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ---------------------------------------------------------
# JSON → DataFrame
# ---------------------------------------------------------
def build_df_from_json(json_path, img_dir):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    id_to_fname = {img['id']: img['file_name'] for img in data['images']}
    records = []
    for ann in data['annotations']:
        file_name = id_to_fname.get(ann['image_id'])
        if not file_name: continue
        img_path = os.path.join(img_dir, file_name)
        if not os.path.exists(img_path): continue
        x, y, w, h = ann['bbox']
        records.append({
            'image_path': img_path,
            'image_id':   os.path.splitext(file_name)[0],
            'category_id': int(ann['category_id']),
            'bbox_x': float(x), 'bbox_y': float(y),
            'bbox_w': float(w), 'bbox_h': float(h),
        })
    return pd.DataFrame(records)

df_train = build_df_from_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
df_val   = build_df_from_json(VAL_JSON_PATH,   VAL_IMG_DIR)

# ---------------------------------------------------------
# 레이블 매핑
# 🚨 Faster R-CNN / RetinaNet: background=0 예약 → 1-based
# 🚨 YOLO: 0-based (아래 YOLO 변환 셀에서 별도 처리)
# ---------------------------------------------------------
unique_cats = sorted(df_train['category_id'].unique())
orig2model  = {cid: i + 1 for i, cid in enumerate(unique_cats)}
num_classes = len(unique_cats) + 1  # background 포함

print(f"✅ 고유 알약 클래스 수 : {len(unique_cats)}종")
print(f"✅ num_classes (bg 포함): {num_classes}  ← 모델 정의 시 이 값 사용")
print(f"✅ Train 이미지 수     : {df_train['image_id'].nunique()}장 / 객체 {len(df_train)}개")
print(f"✅ Val   이미지 수     : {df_val['image_id'].nunique()}장 / 객체 {len(df_val)}개")

# ---------------------------------------------------------
# Dataset 클래스
# ---------------------------------------------------------
class OralDrugDataset(Dataset):
    def __init__(self, df, orig2model, transforms=None):
        self.df        = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms
        self.image_ids  = self.df['image_id'].unique().tolist()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        df_img   = self.df[self.df['image_id'] == image_id]
        image    = Image.open(df_img['image_path'].iloc[0]).convert('RGB')

        boxes, labels = [], []
        for _, row in df_img.iterrows():
            x1, y1 = row['bbox_x'], row['bbox_y']
            x2, y2 = x1 + row['bbox_w'], y1 + row['bbox_h']
            boxes.append([x1, y1, x2, y2])
            labels.append(self.orig2model.get(int(row['category_id']), 1))

        target = {
            'boxes':    torch.tensor(boxes,  dtype=torch.float32),
            'labels':   torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx]),
        }
        if self.transforms:
            image = self.transforms(image)
        return image, target

# ---------------------------------------------------------
# Transforms
# 🚨 T.Normalize 미적용 — Faster R-CNN / RetinaNet은 내부적으로
#    GeneralizedRCNNTransform이 ImageNet 정규화를 직접 수행합니다.
#    DataLoader에서 중복 적용하면 이중 정규화로 학습이 망가집니다.
#
# 💡 추론/시각화 시 역정규화가 필요하다면 아래 함수를 사용하세요.
# def denormalize(tensor, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]):
#     t = tensor.clone()
#     for c, (m, s) in enumerate(zip(mean, std)):
#         t[c] = t[c] * s + m
#     return t.clamp(0, 1)
# ---------------------------------------------------------

train_transforms = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
])
val_transforms = T.Compose([
    T.ToTensor(),
])

train_dataset = OralDrugDataset(df_train, orig2model, train_transforms)
val_dataset   = OralDrugDataset(df_val,   orig2model, val_transforms)

def collate_fn(batch): return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=2, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"\n🚀 DataLoader 준비 완료 | Train {len(train_loader)} steps / Val {len(val_loader)} steps")


✅ 고유 알약 클래스 수 : 73종
✅ num_classes (bg 포함): 74  ← 모델 정의 시 이 값 사용
✅ Train 이미지 수     : 1792장 / 객체 6196개
✅ Val   이미지 수     : 139장 / 객체 431개

🚀 DataLoader 준비 완료 | Train 896 steps / Val 70 steps


In [6]:
############################################################
# 3. YOLO용 레이블 변환기
# - 동일한 JSON(train/val_letterbox.json)을 읽어 txt로 변환
# - 데이터 재분할 불필요, 이미지 파일도 그대로 공유
# 🚨 YOLO 레이블은 0-based (Faster R-CNN의 orig2model과 1 차이)
############################################################
from tqdm.auto import tqdm

def convert_coco_to_yolo(json_path, output_label_dir, cat2yolo=None):
    """
    COCO JSON → YOLO txt 변환기
    출력 형식: <class_id> <cx> <cy> <w> <h>  (모두 0~1 정규화)
    cat2yolo: Train에서 생성한 매핑을 Val에도 그대로 넘겨야 클래스 ID가 일치합니다.
    """
    os.makedirs(output_label_dir, exist_ok=True)

    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    # 외부에서 받은 cat2yolo가 없을 때만 새로 생성 (Train 첫 실행 시)
    # Val은 반드시 Train의 cat2yolo를 받아야 클래스 ID가 일치함
    if cat2yolo is None:
        all_cats = sorted(set(ann['category_id'] for ann in coco['annotations']))
        cat2yolo = {cid: i for i, cid in enumerate(all_cats)}

    id_to_img   = {img['id']: img for img in coco['images']}
    img_to_anns = defaultdict(list)
    for ann in coco['annotations']:
        img_to_anns[ann['image_id']].append(ann)

    for img_id, anns in tqdm(img_to_anns.items(), desc=f"{os.path.basename(output_label_dir)} 변환"):
        img_info = id_to_img[img_id]
        W, H     = img_info['width'], img_info['height']  # 800x800

        txt_name = os.path.splitext(img_info['file_name'])[0] + '.txt'
        with open(os.path.join(output_label_dir, txt_name), 'w') as f:
            for ann in anns:
                x, y, w, h = ann['bbox']
                cx = (x + w / 2) / W
                cy = (y + h / 2) / H
                nw = w / W
                nh = h / H
                cls = cat2yolo[ann['category_id']]
                f.write(f"{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

    print(f"✅ 저장 완료 → {output_label_dir}")
    return cat2yolo


YOLO_TRAIN_LABEL_DIR = os.path.join(BASE_DIR, 'yolo_labels/train')
YOLO_VAL_LABEL_DIR   = os.path.join(BASE_DIR, 'yolo_labels/val')

print("🔄 YOLO 레이블 변환을 시작합니다...\n")
cat2yolo = convert_coco_to_yolo(TRAIN_JSON_PATH, YOLO_TRAIN_LABEL_DIR)
convert_coco_to_yolo(VAL_JSON_PATH, YOLO_VAL_LABEL_DIR, cat2yolo=cat2yolo)

# data.yaml 자동 생성
with open(TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    _coco = json.load(f)
id_to_name  = {cat['id']: cat['name'] for cat in _coco['categories']}
class_names = [id_to_name[cid] for cid in sorted(cat2yolo, key=cat2yolo.get)]

yaml_content = f"""# YOLO data.yaml - Auto-generated by NB05
path: {BASE_DIR}
train: letterbox_images/train
val:   letterbox_images/val
train_labels: yolo_labels/train
val_labels:   yolo_labels/val

nc: {len(class_names)}
names: {class_names}
"""

yaml_path = os.path.join(BASE_DIR, 'data.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"\n✅ data.yaml 생성 완료 → {yaml_path}")
print(f"   nc={len(class_names)}, 이미지 경로·레이블 경로 자동 연결됨")

🔄 YOLO 레이블 변환을 시작합니다...



train 변환:   0%|          | 0/1792 [00:00<?, ?it/s]

✅ 저장 완료 → /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_labels/train


val 변환:   0%|          | 0/139 [00:00<?, ?it/s]

✅ 저장 완료 → /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_labels/val

✅ data.yaml 생성 완료 → /content/drive/MyDrive/data/초급_프로젝트/dataset/data.yaml
   nc=73, 이미지 경로·레이블 경로 자동 연결됨
